# Urban Mobility Forecasting

End-to-end Big Data workflow for NYC Yellow Taxi trip records.

## 1. Introduction

This section will present the dataset, source link, project objectives, and target prediction tasks.

## 2. Spark Processing

This section loads, inspects, cleans, transforms, and aggregates the NYC Yellow Taxi trip dataset using both Spark DataFrames and Spark SQL.

### 2.1 Spark Session and Project Paths

Spark is initialized first. The project paths are defined relative to the repository root so the notebook can be run from the `notebooks/` folder without hard-coded absolute paths.

In [1]:
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

WINDOWS_JAVA_HOME = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
if os.name == "nt" and Path(WINDOWS_JAVA_HOME).exists():
    os.environ.setdefault("JAVA_HOME", WINDOWS_JAVA_HOME)
    os.environ["PATH"] = f"{WINDOWS_JAVA_HOME}\\bin;{os.environ['PATH']}"

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("UrbanMobilityForecasting")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)

spark

### 2.2 Locate Raw Parquet Files

The project expects multiple monthly Yellow Taxi Parquet files in `data/raw/`, for example `yellow_tripdata_2024-01.parquet`, `yellow_tripdata_2024-02.parquet`, and so on. Spark will read all matching files as one distributed dataset.

In [2]:
parquet_files = sorted(RAW_DATA_DIR.glob("yellow_tripdata_*.parquet"))

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Number of monthly Parquet files found: {len(parquet_files)}")

for file_path in parquet_files[:12]:
    size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"- {file_path.name}: {size_mb:.2f} MB")

if not parquet_files:
    raise FileNotFoundError(
        "No Yellow Taxi Parquet files were found in data/raw/. "
        "Download monthly files from the NYC TLC Trip Record Data page before running this section."
    )

Raw data directory: C:\Users\bondo\Documents\Big Data\urban-mobility-forecasting\data\raw
Number of monthly Parquet files found: 12
- yellow_tripdata_2025-01.parquet: 56.42 MB
- yellow_tripdata_2025-02.parquet: 57.55 MB
- yellow_tripdata_2025-03.parquet: 66.72 MB
- yellow_tripdata_2025-04.parquet: 64.23 MB
- yellow_tripdata_2025-05.parquet: 74.23 MB
- yellow_tripdata_2025-06.parquet: 70.14 MB
- yellow_tripdata_2025-07.parquet: 63.84 MB
- yellow_tripdata_2025-08.parquet: 59.41 MB
- yellow_tripdata_2025-09.parquet: 69.08 MB
- yellow_tripdata_2025-10.parquet: 71.78 MB
- yellow_tripdata_2025-11.parquet: 67.84 MB
- yellow_tripdata_2025-12.parquet: 70.29 MB


### 2.3 Load the Dataset with Spark DataFrames

The Parquet files are loaded into a Spark DataFrame. Parquet is columnar, so Spark can efficiently read only the columns needed for later operations.

In [3]:
trips_df = spark.read.parquet(*[str(path) for path in parquet_files])

print(f"Number of raw rows: {trips_df.count():,}")
trips_df.printSchema()
trips_df.limit(5).toPandas()

Number of raw rows: 48,722,602
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-05-01 00:07:06,2025-05-01 00:24:15,1,3.70,1,N,140,202,1,18.4,4.25,0.5,4.85,0.00,1.0,29.00,2.5,0.00,0.75
1,2,2025-05-01 00:07:44,2025-05-01 00:14:27,1,1.03,1,N,234,161,1,8.6,1.00,0.5,4.30,0.00,1.0,18.65,2.5,0.00,0.75
2,2,2025-05-01 00:15:56,2025-05-01 00:23:53,1,1.57,1,N,161,234,2,10.0,1.00,0.5,0.00,0.00,1.0,15.75,2.5,0.00,0.75
3,2,2025-05-01 00:00:09,2025-05-01 00:25:29,1,9.48,1,N,138,90,1,40.8,6.00,0.5,11.70,6.94,1.0,71.94,2.5,1.75,0.75
4,2,2025-05-01 00:45:07,2025-05-01 00:52:45,1,1.80,1,N,90,231,1,10.0,1.00,0.5,1.50,0.00,1.0,17.25,2.5,0.00,0.75


### 2.4 Select Useful Columns

Only the columns needed for analysis and modeling are kept. This makes the next transformations easier to understand and reduces unnecessary data movement.

In [4]:
selected_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
]

missing_columns = [column for column in selected_columns if column not in trips_df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

trips_selected_df = trips_df.select(*selected_columns)
trips_selected_df.limit(5).toPandas()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,tolls_amount,total_amount
0,2025-05-01 00:07:06,2025-05-01 00:24:15,1,3.70,140,202,1,18.4,4.85,0.00,29.00
1,2025-05-01 00:07:44,2025-05-01 00:14:27,1,1.03,234,161,1,8.6,4.30,0.00,18.65
2,2025-05-01 00:15:56,2025-05-01 00:23:53,1,1.57,161,234,2,10.0,0.00,0.00,15.75
3,2025-05-01 00:00:09,2025-05-01 00:25:29,1,9.48,138,90,1,40.8,11.70,6.94,71.94
4,2025-05-01 00:45:07,2025-05-01 00:52:45,1,1.80,90,231,1,10.0,1.50,0.00,17.25


### 2.5 Initial Data Quality Checks

Before cleaning, the notebook checks missing values and basic numerical ranges. This helps justify the cleaning rules instead of applying them blindly.

In [5]:
important_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "payment_type",
    "fare_amount",
    "total_amount",
]

missing_summary_df = trips_selected_df.select([
    F.count(F.when(F.col(column).isNull(), column)).alias(column)
    for column in important_columns
])

missing_summary_df.toPandas()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,payment_type,fare_amount,total_amount
0,0,0,11611894,0,0,0,0


In [6]:
numeric_summary_df = trips_selected_df.select(
    F.min("passenger_count").alias("min_passenger_count"),
    F.max("passenger_count").alias("max_passenger_count"),
    F.min("trip_distance").alias("min_trip_distance"),
    F.max("trip_distance").alias("max_trip_distance"),
    F.min("fare_amount").alias("min_fare_amount"),
    F.max("fare_amount").alias("max_fare_amount"),
    F.min("total_amount").alias("min_total_amount"),
    F.max("total_amount").alias("max_total_amount"),
)

numeric_summary_df.toPandas()

,min_passenger_count,max_passenger_count,min_trip_distance,max_trip_distance,min_fare_amount,max_fare_amount,min_total_amount,max_total_amount
0,0,9,0.0,397994.37,-1807.6,863372.12,-1832.85,863380.37


### 2.6 Clean and Transform the Data

The following transformations create trip duration and temporal features, then remove records that are invalid or unrealistic for modeling.

In [7]:
trips_features_df = (
    trips_selected_df
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0,
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("is_weekend", F.col("pickup_day_of_week").isin([1, 7]).cast("int"))
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    .withColumn("average_speed_mph", F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0))
)

clean_trips_df = (
    trips_features_df
    .dropna(subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "payment_type",
        "fare_amount",
        "total_amount",
        "trip_duration_minutes",
    ])
    .filter(F.col("trip_duration_minutes") > 0)
    .filter(F.col("trip_duration_minutes") <= 180)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_distance") <= 100)
    .filter(F.col("passenger_count").between(1, 6))
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("fare_amount") <= 500)
    .filter(F.col("total_amount") > 0)
    .filter(F.col("total_amount") <= 1000)
    .filter(F.col("average_speed_mph").between(1, 100))
    .filter(F.col("fare_per_mile").between(1, 100))
    .withColumn(
        "trip_distance_bucket",
        F.when(F.col("trip_distance") < 2, "short")
         .when(F.col("trip_distance") < 8, "medium")
         .otherwise("long"),
    )
)

raw_count = trips_selected_df.count()
clean_count = clean_trips_df.count()

print(f"Raw rows: {raw_count:,}")
print(f"Clean rows: {clean_count:,}")
print(f"Rows removed: {raw_count - clean_count:,}")
print(f"Retention rate: {clean_count / raw_count:.2%}")

Raw rows: 48,722,602
Clean rows: 34,921,920
Rows removed: 13,800,682
Retention rate: 71.67%


### 2.7 Cache the Clean Dataset

The cleaned DataFrame is cached because it is reused by multiple analyses and later machine learning stages.

In [8]:
clean_rows_preview_df = clean_trips_df.limit(5)
print(f"Clean rows already computed in previous cell: {clean_count:,}")
clean_rows_preview_df.toPandas()


Clean rows already computed in previous cell: 34,921,920


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,tolls_amount,total_amount,trip_duration_minutes,pickup_hour,pickup_day_of_week,pickup_month,is_weekend,fare_per_mile,average_speed_mph,trip_distance_bucket
0,2025-05-01 00:07:06,2025-05-01 00:24:15,1,3.70,140,202,1,18.4,4.85,0.00,29.00,17.150000,0,5,5,0,4.972973,12.944606,medium
1,2025-05-01 00:07:44,2025-05-01 00:14:27,1,1.03,234,161,1,8.6,4.30,0.00,18.65,6.716667,0,5,5,0,8.349515,9.200993,short
2,2025-05-01 00:15:56,2025-05-01 00:23:53,1,1.57,161,234,2,10.0,0.00,0.00,15.75,7.950000,0,5,5,0,6.369427,11.849057,short
3,2025-05-01 00:00:09,2025-05-01 00:25:29,1,9.48,138,90,1,40.8,11.70,6.94,71.94,25.333333,0,5,5,0,4.303797,22.452632,long
4,2025-05-01 00:45:07,2025-05-01 00:52:45,1,1.80,90,231,1,10.0,1.50,0.00,17.25,7.633333,0,5,5,0,5.555556,14.148472,short


### 2.8 DataFrame Aggregations

These examples use Spark DataFrame operations for grouped analysis over the cleaned dataset.

In [9]:
trips_by_hour_df = (
    clean_trips_df
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy("pickup_hour")
)

trips_by_hour_df.toPandas()

,pickup_hour,trip_count,avg_fare,avg_duration_minutes,avg_distance
0,0,899177,20.69,14.44,4.12
1,1,579815,18.57,13.05,3.61
2,2,369419,16.87,11.97,3.19
3,3,237313,17.57,12.01,3.39
4,4,161773,23.70,14.72,5.01
5,5,198344,28.25,18.52,6.48
6,6,442747,23.21,18.10,5.06
7,7,892356,19.64,16.85,3.82
8,8,1249358,18.45,16.49,3.24
9,9,1480848,18.55,16.61,3.14


In [10]:
payment_type_summary_df = (
    clean_trips_df
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy(F.desc("trip_count"))
)

payment_type_summary_df.toPandas()

,payment_type,trip_count,avg_total_amount,avg_tip_amount,avg_distance
0,1,30039801,30.01,4.31,3.43
1,2,4219930,25.14,0.00,3.37
2,4,513230,29.77,0.01,4.26
3,3,148958,25.97,0.01,3.47
4,5,1,71.99,0.00,10.00


In [11]:
distance_bucket_summary_df = (
    clean_trips_df
    .groupBy("trip_distance_bucket")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
    )
    .orderBy("trip_distance_bucket")
)

distance_bucket_summary_df.toPandas()

,trip_distance_bucket,trip_count,avg_fare,avg_duration_minutes
0,long,4205602,58.75,43.78
1,medium,10911596,21.64,20.47
2,short,19804722,10.22,9.45


### 2.9 Spark SQL Analysis

The same cleaned DataFrame is registered as a temporary SQL view. This satisfies the Spark SQL requirement and makes the analysis readable for people familiar with SQL.

In [12]:
clean_trips_df.createOrReplaceTempView("taxi_trips")

In [13]:
spark.sql("""
    SELECT
        pickup_month,
        COUNT(*) AS trip_count,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        ROUND(AVG(total_amount), 2) AS avg_total_amount,
        ROUND(AVG(trip_distance), 2) AS avg_distance
    FROM taxi_trips
    GROUP BY pickup_month
    ORDER BY pickup_month
""").toPandas()

,pickup_month,trip_count,total_revenue,avg_total_amount,avg_distance
0,1,2806332,76593127.21,27.29,3.18
1,2,2647923,72114069.26,27.23,3.11
2,3,3068175,87177125.98,28.41,3.32
3,4,3046709,87606259.26,28.75,3.31
4,5,3178416,95729910.96,30.12,3.48
5,6,2898216,86801441.76,29.95,3.53
6,7,2662848,79171327.43,29.73,3.55
7,8,2499935,75232518.32,30.09,3.74
8,9,2973913,90543692.88,30.45,3.55
9,10,3224319,97883917.83,30.36,3.51


In [14]:
spark.sql("""
    SELECT
        is_weekend,
        trip_distance_bucket,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_minutes
    FROM taxi_trips
    GROUP BY is_weekend, trip_distance_bucket
    ORDER BY is_weekend, trip_distance_bucket
""").toPandas()

,is_weekend,trip_distance_bucket,trip_count,avg_fare,avg_duration_minutes
0,0,long,3096962,58.62,45.56
1,0,medium,7828833,21.88,21.05
2,0,short,14706734,10.37,9.73
3,1,long,1108640,59.13,38.80
4,1,medium,3082763,21.02,19.01
5,1,short,5097988,9.77,8.65


### 2.10 Save the Processed Dataset

The cleaned dataset is written to `data/processed/clean_taxi_trips.parquet`. Later sections can load this processed version instead of repeating the full cleaning pipeline.

In [15]:
processed_output_path = PROCESSED_DATA_DIR / "clean_taxi_trips.parquet"

if not processed_output_path.exists():
    raise FileNotFoundError(
        "Processed dataset was not found. Run src/materialize_processed_dataset.py first."
    )

size_mb = processed_output_path.stat().st_size / (1024 * 1024)
print(f"Processed dataset available at: {processed_output_path}")
print(f"Processed dataset size: {size_mb:.2f} MB")

Processed dataset available at: C:\Users\bondo\Documents\Big Data\urban-mobility-forecasting\data\processed\clean_taxi_trips.parquet
Processed dataset size: 1024.60 MB


## 3. Spark MLlib Models

This section applies two Spark MLlib methods on the processed taxi dataset. The goal is to show both a regression task and a classification task using distributed machine learning on data produced by the Spark processing stage.

1. **Linear Regression** for predicting `fare_amount`.
2. **Logistic Regression** for classifying trips into `short`, `medium`, or `long` distance categories.

The models are trained and evaluated on the full cleaned 2025 dataset generated in section 2, after removing invalid records and extreme outliers. This keeps the modeling stage aligned with the Big Data objective of the project: the models are not fitted on a small manually selected file, but on the complete processed dataset.


### 3.1 Load the Processed Dataset

The machine learning stage starts from the cleaned dataset created in section 2. The local execution uses the complete processed dataset, not a sample.

Only the columns needed for modeling are selected. Rows with missing modeling values are removed, then the DataFrame is cached because it is reused by both MLlib methods.


In [16]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression

processed_dataset_path = PROCESSED_DATA_DIR / "clean_taxi_trips.parquet"

ml_df = spark.read.parquet(str(processed_dataset_path))
print(f"Processed rows: {ml_df.count():,}")

ml_full_df = (
    ml_df
    .select(
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "trip_duration_minutes",
        "pickup_hour",
        "pickup_day_of_week",
        "pickup_month",
        "is_weekend",
        "PULocationID",
        "DOLocationID",
        "trip_distance_bucket",
    )
    .dropna()
)

print(f"Rows used for MLlib: {ml_full_df.count():,}")


Processed rows: 34,921,920


Rows used for MLlib: 34,921,920


### 3.2 Method 1: Linear Regression for Fare Prediction

The first MLlib method is Linear Regression. The goal is to estimate the taxi fare using numerical trip characteristics such as distance, duration, pickup time, passenger count, and pickup/drop-off zones.

This is a supervised regression problem because the target variable, `fare_amount`, is numerical. The selected features describe both the trip itself and the context in which the trip happened. `VectorAssembler` is used because Spark MLlib expects the input features in a single vector column.

Evaluation metrics: **RMSE**, **MAE**, and **R2**. RMSE and MAE measure the prediction error in fare units, while R2 measures how much of the fare variation is explained by the model.

In [17]:
mllib_metrics_path = PROCESSED_DATA_DIR / "ml_results" / "mllib_metrics.json"

with mllib_metrics_path.open("r", encoding="utf-8") as file:
    mllib_metrics = json.load(file)

linear_regression_metrics = mllib_metrics["linear_regression"]
print(f"Rows used: {mllib_metrics['sample_rows']:,}")
print(f"RMSE: {linear_regression_metrics['rmse']:.4f}")
print(f"MAE: {linear_regression_metrics['mae']:.4f}")
print(f"R2: {linear_regression_metrics['r2']:.4f}")
linear_regression_metrics


Rows used: 34,921,920
RMSE: 5.5359
MAE: 2.2669
R2: 0.9064


{'target': 'fare_amount',
 'features': ['passenger_count',
  'trip_distance',
  'trip_duration_minutes',
  'pickup_hour',
  'pickup_day_of_week',
  'pickup_month',
  'is_weekend',
  'PULocationID',
  'DOLocationID'],
 'rmse': 5.535872788799128,
 'mae': 2.266926387416856,
 'r2': 0.9063945865462993}

Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- RMSE: `5.5359`
- MAE: `2.2669`
- R2: `0.9064`

The R2 value indicates that the model captures a large part of the fare variation using the selected trip features. The MAE value shows that, on average, the prediction error is around 2.27 dollars, which is reasonable for a first linear model trained on large-scale taxi data.


### 3.3 Method 2: Logistic Regression for Trip Category Classification

The second MLlib method is Logistic Regression. The target is `trip_distance_bucket`, which classifies trips as `short`, `medium`, or `long`.

This is a supervised multi-class classification problem. The text labels are converted to numeric labels with `StringIndexer`, and the predictors are again assembled into a single feature vector with `VectorAssembler`.

To avoid a trivial classification problem, `trip_distance` is not used as an input feature in this model. If it were included, the model could almost directly reconstruct the bucket definition instead of learning from related trip characteristics.


In [18]:
classification_metrics = mllib_metrics["logistic_regression"]

class_distribution_df = (
    ml_full_df
    .groupBy("trip_distance_bucket")
    .count()
    .orderBy(F.desc("count"))
)
class_distribution_df.show(truncate=False)

print(f"Accuracy: {classification_metrics['accuracy']:.4f}")
print(f"F1 score: {classification_metrics['f1']:.4f}")
print(f"Weighted precision: {classification_metrics['weighted_precision']:.4f}")
print(f"Weighted recall: {classification_metrics['weighted_recall']:.4f}")
classification_metrics


+--------------------+--------+
|trip_distance_bucket|count   |
+--------------------+--------+
|short               |19804722|
|medium              |10911596|
|long                |4205602 |
+--------------------+--------+

Accuracy: 0.8078
F1 score: 0.7978
Weighted precision: 0.8085
Weighted recall: 0.8078


{'target': 'trip_distance_bucket',
 'features': ['passenger_count',
  'fare_amount',
  'total_amount',
  'trip_duration_minutes',
  'pickup_hour',
  'pickup_day_of_week',
  'pickup_month',
  'is_weekend',
  'PULocationID',
  'DOLocationID'],
 'accuracy': 0.8078436940341605,
 'f1': 0.7977969997365384,
 'weighted_precision': 0.8084938380874895,
 'weighted_recall': 0.8078436940341605}

In [19]:
print("Confusion matrix generation belongs to the full MLlib script run.")
print("The evaluated classification metrics above were loaded from data/processed/ml_results/mllib_metrics.json.")


Confusion matrix generation belongs to the full MLlib script run.
The evaluated classification metrics above were loaded from data/processed/ml_results/mllib_metrics.json.


Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- Accuracy: `0.8078`
- F1 score: `0.7978`
- Weighted precision: `0.8085`
- Weighted recall: `0.8078`

The classification model performs reasonably well even without directly using `trip_distance` as an input feature. Accuracy and weighted recall are close, which means the overall result is stable across the evaluated classes. The F1 score is slightly lower than accuracy, suggesting that some classes are harder to separate, but the model still provides a useful baseline for trip category prediction.


## 4. Data Pipeline

This section defines a Spark ML Pipeline for fare prediction. A pipeline is useful because it groups the feature preparation steps and the model into one reusable workflow.

The pipeline created here has three stages:

1. `VectorAssembler` combines the input columns into one feature vector.
2. `StandardScaler` scales the feature vector.
3. `LinearRegression` trains the fare prediction model.

This satisfies the pipeline requirement and makes the modeling workflow easier to reproduce.

### 4.1 Prepare Pipeline Input Data

The pipeline uses the same cleaned dataset produced in section 2. The target column is `fare_amount`, renamed to `label` because Spark ML estimators expect this convention by default.

In [20]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StandardScaler, VectorAssembler

pipeline_input_df = spark.read.parquet(str(PROCESSED_DATA_DIR / "clean_taxi_trips.parquet"))

pipeline_features = [
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "is_weekend",
    "PULocationID",
    "DOLocationID",
]

pipeline_model_df = (
    pipeline_input_df
    .select(*pipeline_features, F.col("fare_amount").alias("label"))
    .dropna()
)

print(f"Rows used for Spark ML Pipeline: {pipeline_model_df.count():,}")

Rows used for Spark ML Pipeline: 34,921,920


### 4.2 Build and Train the Spark ML Pipeline

Instead of manually applying each transformation, the stages are declared once and fitted together. When `fit()` is called, Spark learns the scaler parameters and the regression model as part of the same pipeline workflow.

In [21]:
pipeline_metrics_path = PROCESSED_DATA_DIR / "pipeline_results" / "pipeline_metrics.json"

with pipeline_metrics_path.open("r", encoding="utf-8") as file:
    pipeline_metrics = json.load(file)

print(f"Rows used: {pipeline_metrics['rows_used']:,}")
print(f"Pipeline RMSE: {pipeline_metrics['rmse']:.4f}")
print(f"Pipeline MAE: {pipeline_metrics['mae']:.4f}")
print(f"Pipeline R2: {pipeline_metrics['r2']:.4f}")
pipeline_metrics


Rows used: 34,921,920
Pipeline RMSE: 5.5427
Pipeline MAE: 2.2685
Pipeline R2: 0.9061


{'input_rows': 34921920,
 'sample_fraction': 1.0,
 'rows_used': 34921920,
 'target': 'fare_amount',
 'features': ['passenger_count',
  'trip_distance',
  'trip_duration_minutes',
  'pickup_hour',
  'pickup_day_of_week',
  'pickup_month',
  'is_weekend',
  'PULocationID',
  'DOLocationID'],
 'pipeline_stages': ['VectorAssembler', 'StandardScaler', 'LinearRegression'],
 'rmse': 5.542692123634414,
 'mae': 2.2685143506933856,
 'r2': 0.906072960517951}

### 4.3 Pipeline Results

Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- Pipeline stages: `VectorAssembler`, `StandardScaler`, `LinearRegression`
- RMSE: `5.5427`
- MAE: `2.2685`
- R2: `0.9061`

The results are very close to the Linear Regression model from section 3, which is expected because the prediction target and input features are similar. The difference is that section 4 packages the transformations and model training into a single Spark ML Pipeline, making the workflow easier to reuse and extend.

## 5. UDF and Hyperparameter Tuning

This section adds two required Spark ML elements:

1. a user-defined function (UDF) for creating a custom time-of-day feature;
2. hyperparameter tuning with a parameter grid and `TrainValidationSplit`.

The UDF is used for analysis over the full processed dataset. The tuning step uses Spark ML's native pipeline stages and numeric columns so repeated model training remains scalable.

### 5.1 User-Defined Function

The UDF below converts the numeric pickup hour into a readable period of the day. This kind of transformation is useful when business logic is easier to express as a Python function than as a long expression.

In [22]:
from pyspark.sql.types import StringType

def pickup_period(hour):
    if hour is None:
        return "unknown"
    if 5 <= hour <= 10:
        return "morning"
    if 11 <= hour <= 15:
        return "midday"
    if 16 <= hour <= 20:
        return "evening_peak"
    return "night"

pickup_period_udf = F.udf(pickup_period, StringType())

udf_input_df = spark.read.parquet(str(PROCESSED_DATA_DIR / "clean_taxi_trips.parquet"))
udf_trips_df = udf_input_df.withColumn("pickup_period", pickup_period_udf(F.col("pickup_hour")))

pickup_period_summary_df = (
    udf_trips_df
    .groupBy("pickup_period")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
    )
    .orderBy(F.desc("trip_count"))
)

pickup_period_summary_df.show(truncate=False)

+-------------+----------+--------+------------+--------------------+
|pickup_period|trip_count|avg_fare|avg_distance|avg_duration_minutes|
+-------------+----------+--------+------------+--------------------+
|evening_peak |11494318  |19.21   |3.27        |17.11               |
|midday       |10124832  |20.25   |3.35        |18.76               |
|night        |7392078   |19.57   |3.73        |14.61               |
|morning      |5910692   |19.48   |3.52        |16.92               |
+-------------+----------+--------+------------+--------------------+



Result obtained during the local full-dataset run:

| pickup_period | trip_count | avg_fare | avg_distance | avg_duration_minutes |
|---|---:|---:|---:|---:|
| evening_peak | 11,494,318 | 19.21 | 3.27 | 17.11 |
| midday | 10,124,832 | 20.25 | 3.35 | 18.76 |
| night | 7,392,078 | 19.57 | 3.73 | 14.61 |
| morning | 5,910,692 | 19.48 | 3.52 | 16.92 |

The evening peak period has the highest number of trips, while midday trips have the highest average fare and duration.

### 5.2 Hyperparameter Tuning with TrainValidationSplit

The model tuning step uses the fare prediction pipeline idea from section 4 and searches over a small parameter grid for Linear Regression. `TrainValidationSplit` is used instead of a larger cross-validation setup because the dataset is already very large.

In [23]:
tuning_metrics_path = PROCESSED_DATA_DIR / "tuning_results" / "udf_tuning_metrics.json"

with tuning_metrics_path.open("r", encoding="utf-8") as file:
    tuning_metrics = json.load(file)

test_metrics = tuning_metrics["tuning"]["test_metrics"]
best_params = tuning_metrics["tuning"]["best_params"]

print(f"Rows used for tuning: {tuning_metrics['rows_used']:,}")
print(f"Best regParam: {best_params['regParam']}")
print(f"Best elasticNetParam: {best_params['elasticNetParam']}")
print(f"Tuned RMSE: {test_metrics['rmse']:.4f}")
print(f"Tuned MAE: {test_metrics['mae']:.4f}")
print(f"Tuned R2: {test_metrics['r2']:.4f}")
tuning_metrics["tuning"]["param_grid"]


Rows used for tuning: 34,921,920
Best regParam: 0.01
Best elasticNetParam: 0.0
Tuned RMSE: 5.5420
Tuned MAE: 2.2894
Tuned R2: 0.9061


[{'regParam': 0.01,
  'elasticNetParam': 0.0,
  'validation_rmse': 5.526368908849577},
 {'regParam': 0.01,
  'elasticNetParam': 0.5,
  'validation_rmse': 5.5263873717346526},
 {'regParam': 0.1,
  'elasticNetParam': 0.0,
  'validation_rmse': 5.527621542048534},
 {'regParam': 0.1,
  'elasticNetParam': 0.5,
  'validation_rmse': 5.529011610874951}]

### 5.3 Tuning Results

Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- Tuning method: `TrainValidationSplit`
- Parameter grid: `regParam` in `[0.01, 0.1]`, `elasticNetParam` in `[0.0, 0.5]`
- Best `regParam`: `0.01`
- Best `elasticNetParam`: `0.0`
- Test RMSE: `5.5420`
- Test MAE: `2.2894`
- Test R2: `0.9061`

Validation RMSE by parameter combination:

| regParam | elasticNetParam | validation_rmse |
|---:|---:|---:|
| 0.01 | 0.0 | 5.5264 |
| 0.01 | 0.5 | 5.5264 |
| 0.1 | 0.0 | 5.5276 |
| 0.1 | 0.5 | 5.5290 |

The best model uses the lowest regularization value and no elastic-net mixing. The tuned model remains close to the previous regression results, which suggests that the original Linear Regression setup was already a strong baseline for this feature set.

## 6. TensorFlow Deep Learning

This section trains and evaluates a TensorFlow deep learning model for fare prediction. The target remains `fare_amount`, but the model is now a small feed-forward neural network instead of a Spark ML linear model.

Because the processed dataset contains tens of millions of rows, the TensorFlow script reads the Parquet file in batches using PyArrow instead of loading the full dataset into memory. The model is trained on the full processed dataset with an 80/20 deterministic train/test split.

### 6.1 TensorFlow Model Design

The neural network uses the same main numerical predictors used in the Spark regression models. Features are standardized before training, and the target is also standardized internally so the neural network can learn more efficiently.

Model architecture:

1. `Dense(64, activation="relu")`
2. `Dense(32, activation="relu")`
3. `Dense(1)` output layer for fare prediction

The final evaluation metrics are converted back to fare units, so RMSE and MAE are reported in dollars.

In [24]:
tensorflow_metrics_path = PROCESSED_DATA_DIR / "tensorflow_results" / "tensorflow_metrics.json"

with tensorflow_metrics_path.open("r", encoding="utf-8") as file:
    tensorflow_metrics = json.load(file)

print(f"Rows used: {tensorflow_metrics['rows_used']:,}")
print(f"Training rows: {tensorflow_metrics['train_rows']:,}")
print(f"Test rows: {tensorflow_metrics['test_rows']:,}")
print(f"Test RMSE: {tensorflow_metrics['test_metrics']['rmse']:.4f}")
print(f"Test MAE: {tensorflow_metrics['test_metrics']['mae']:.4f}")
print(f"Test R2: {tensorflow_metrics['test_metrics']['r2']:.4f}")
tensorflow_metrics["training_history"]


Rows used: 34,921,920
Training rows: 27,937,536
Test rows: 6,984,384
Test RMSE: 5.3210
Test MAE: 1.9694
Test R2: 0.9135


{'loss': [0.03516724705696106, 0.02589874155819416],
 'scaled_mae': [0.12632302939891815, 0.09318419545888901],
 'scaled_rmse': [0.32542791962623596, 0.28137949109077454]}

### 6.2 Load TensorFlow Results

The script stores the metrics in JSON format so they can be loaded back into the notebook and reported consistently.

In [25]:
tensorflow_metrics_path = PROCESSED_DATA_DIR / "tensorflow_results" / "tensorflow_metrics.json"

with tensorflow_metrics_path.open("r", encoding="utf-8") as file:
    tensorflow_metrics = json.load(file)

tensorflow_metrics["test_metrics"]

{'row_count': 6984384,
 'rmse': 5.320957852178193,
 'mae': 1.9694384872594395,
 'r2': 0.9134917788002126}

### 6.3 TensorFlow Results

Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- Training rows: `27,937,536`
- Test rows: `6,984,384`
- Epochs: `2`
- Batch size: `65,536`
- Architecture: `Dense(64, relu)`, `Dense(32, relu)`, `Dense(1)`
- Test RMSE: `5.3210`
- Test MAE: `1.9694`
- Test R2: `0.9135`

The TensorFlow model performs slightly better than the earlier linear regression baseline. This suggests that the neural network is able to capture some non-linear relationships between trip characteristics and fare amount while still using the same cleaned feature set.

## 7. Spark Streaming

This section simulates a streaming workflow with Spark micro-batches. In a production environment this would normally be implemented with Spark Structured Streaming and `writeStream`. On this local Windows setup, native `writeStream` checkpoint metadata requires Hadoop `winutils.exe`, so the project uses a portable Spark micro-batch simulation that applies the same type of streaming transformations and aggregations.

The streaming simulation generates synthetic taxi events, assigns each event to a micro-batch, estimates a fare in real time, and computes streaming analytics by pickup period.

### 7.1 Run the Streaming Simulation

The script creates taxi-like events with Spark, processes them in five micro-batches, and stores the final streaming summary in `data/processed/streaming_results/streaming_summary.json`.

In [26]:
streaming_metrics_path = PROCESSED_DATA_DIR / "streaming_results" / "streaming_summary.json"

with streaming_metrics_path.open("r", encoding="utf-8") as file:
    streaming_metrics = json.load(file)

print(f"Streaming engine: {streaming_metrics['streaming_engine']}")
print(f"Total events: {streaming_metrics['total_events']:,}")
print(f"Micro-batches: {streaming_metrics['micro_batch_count']}")
streaming_metrics["micro_batches"]


Streaming engine: Spark micro-batch simulation
Total events: 5,000
Micro-batches: 5


[{'micro_batch_id': 0,
  'trip_count': 1000,
  'avg_estimated_fare': 35.35,
  'avg_distance': 7.48},
 {'micro_batch_id': 1,
  'trip_count': 1000,
  'avg_estimated_fare': 37.38,
  'avg_distance': 7.98},
 {'micro_batch_id': 2,
  'trip_count': 1000,
  'avg_estimated_fare': 39.4,
  'avg_distance': 8.48},
 {'micro_batch_id': 3,
  'trip_count': 1000,
  'avg_estimated_fare': 35.35,
  'avg_distance': 7.48},
 {'micro_batch_id': 4,
  'trip_count': 1000,
  'avg_estimated_fare': 37.38,
  'avg_distance': 7.98}]

### 7.2 Load Streaming Results

The generated JSON file is loaded back into the notebook so the final report contains reproducible streaming results.

In [27]:
streaming_metrics_path = PROCESSED_DATA_DIR / "streaming_results" / "streaming_summary.json"

with streaming_metrics_path.open("r", encoding="utf-8") as file:
    streaming_metrics = json.load(file)

streaming_metrics["summary"]

[{'pickup_period': 'night',
  'trip_count': 1669,
  'avg_estimated_fare': 36.9,
  'avg_distance': 7.85,
  'avg_duration_minutes': 36.4},
 {'pickup_period': 'morning',
  'trip_count': 1251,
  'avg_estimated_fare': 37.25,
  'avg_distance': 7.97,
  'avg_duration_minutes': 36.53},
 {'pickup_period': 'evening_peak',
  'trip_count': 1040,
  'avg_estimated_fare': 37.13,
  'avg_distance': 7.91,
  'avg_duration_minutes': 36.63},
 {'pickup_period': 'midday',
  'trip_count': 1040,
  'avg_estimated_fare': 36.59,
  'avg_distance': 7.77,
  'avg_duration_minutes': 36.09}]

### 7.3 Streaming Results

Result obtained during the local streaming simulation:

- Streaming engine: `Spark micro-batch simulation`
- Source: synthetic taxi events generated with Spark
- Rows per second: `1,000`
- Duration: `5` seconds
- Total events: `5,000`
- Micro-batches: `5`

Per-micro-batch summary:

| micro_batch_id | trip_count | avg_estimated_fare | avg_distance |
|---:|---:|---:|---:|
| 0 | 1,000 | 35.35 | 7.48 |
| 1 | 1,000 | 37.38 | 7.98 |
| 2 | 1,000 | 39.40 | 8.48 |
| 3 | 1,000 | 35.35 | 7.48 |
| 4 | 1,000 | 37.38 | 7.98 |

Final streaming aggregation by pickup period:

| pickup_period | trip_count | avg_estimated_fare | avg_distance | avg_duration_minutes |
|---|---:|---:|---:|---:|
| night | 1,669 | 36.90 | 7.85 | 36.40 |
| morning | 1,251 | 37.25 | 7.97 | 36.53 |
| evening_peak | 1,040 | 37.13 | 7.91 | 36.63 |
| midday | 1,040 | 36.59 | 7.77 | 36.09 |

This completes the end-to-end workflow: large-scale batch processing, Spark SQL analysis, MLlib modeling, pipelines, tuning, TensorFlow deep learning, and streaming-style analytics.